# MLlib hands-on: RandomForest para predecir la severidad del retraso

## Descripción de las variables

El dataset está compuesto por las siguientes variables:

1. **Year** 2008
2. **Month** 1
3. **DayofMonth** 1-31
4. **DayOfWeek** 1 (Monday) - 7 (Sunday)
5. **DepTime** hora real de salida (local, hhmm)
6. **CRSDepTime** hora prevista de salida (local, hhmm)
7. **ArrTime** hora real de llegada (local, hhmm)
8. **CRSArrTime** hora prevista de llegada (local, hhmm)
9. **UniqueCarrier** código del aparato
10. **FlightNum** número de vuelo
11. **TailNum** identificador de cola: aircraft registration, unique aircraft identifier
12. **ActualElapsedTime** tiempo real invertido en el vuelo
13. **CRSElapsedTime** en minutos
14. **AirTime** en minutos
15. **ArrDelay** retraso a la llegada, en minutos: se considera que un vuelo ha llegado "on time" si aterrizó menos de 15 minutos más tarde de la hora prevista en el Computerized Reservations Systems (CRS).
16. **DepDelay** retraso a la salida, en minutos
17. **Origin** código IATA del aeropuerto de origen
18. **Dest** código IATA del aeropuerto de destino
19. **Distance** en millas
20. **TaxiIn** taxi in time, in minutes
21. **TaxiOut** taxi out time in minutes
22. **Cancelled** *si el vuelo fue cancelado (1 = sí, 0 = no)
23. **CancellationCode** razón de cancelación (A = aparato, B = tiempo atmosférico, C = NAS, D = seguridad)
24. **Diverted** *si el vuelo ha sido desviado (1 = sí, 0 = no)
25. **CarrierDelay** en minutos: El retraso del transportista está bajo el control del transportista aéreo. Ejemplos de sucesos que pueden determinar el retraso del transportista son: limpieza de la aeronave, daño de la aeronave, espera de la llegada de los pasajeros o la tripulación de conexión, equipaje, impacto de un pájaro, carga de equipaje, servicio de comidas, computadora, equipo del transportista, problemas legales de la tripulación (descanso del piloto o acompañante) , daños por mercancías peligrosas, inspección de ingeniería, abastecimiento de combustible, pasajeros discapacitados, tripulación retrasada, servicio de inodoros, mantenimiento, ventas excesivas, servicio de agua potable, denegación de viaje a pasajeros en mal estado, proceso de embarque muy lento, equipaje de mano no válido, retrasos de peso y equilibrio.
26. **WeatherDelay** en minutos: causado por condiciones atmosféricas extremas o peligrosas, previstas o que se han manifestado antes del despegue, durante el viaje, o a la llegada.
27. **NASDelay** en minutos: retraso causado por el National Airspace System (NAS) por motivos como condiciones meteorológicas (perjudiciales pero no extremas), operaciones del aeropuerto, mucho tráfico aéreo, problemas con los controladores aéreos, etc.
28. **SecurityDelay** en minutos: causado por la evacuación de una terminal, re-embarque de un avión debido a brechas en la seguridad, fallos en dispositivos del control de seguridad, colas demasiado largas en el control de seguridad, etc.
29. **LateAircraftDelay** en minutos: debido al propio retraso del avión al llegar, problemas para conseguir aterrizar en un aeropuerto a una hora más tardía de la que estaba prevista.

In [0]:
# Leemos el dataset de HDFS. Esta operación todavía no hace la lectura hace una pasada 
# sobre los datos para inferir el esquema
flightsDF = spark.read.option("header", "true")\
                      .option("inferSchema", "true")\
                      .csv("abfss://datos@cursosparkucm.dfs.core.windows.net/flights-jan-apr-2018.csv")

In [0]:
flightsDF.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- TaxiIn: string (nullable = true)
 |-- TaxiOut: string (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true)
 |-- Ca

Vamos a utilizar algunos transformadores habituales, y un algoritmo RandomForest que es un estimador. Utilizaremos:

* StringIndexer para convertir variables tipo String en variables categóricas pero cuyos valores son números reales con la parte decimal a 0, tal como necesitan los algoritmos de Spark.
* Bucketizer para discretizar la columna de ArrDelay sin dar nombre a las categorías, solo numeros. Será nuestra variable target.
* VectorAssembler para unir las columnas de las features en una sola de tipo vector
* RandomForest que es un estimador, como algoritmo de predicción de la severidad
* Pipeline, que es un estimador y que incluirá todos los elementos anteriores.

In [0]:
flightsDF.select("Origin").distinct().count()

356

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import Bucketizer, StringIndexer, VectorAssembler, StringIndexerModel
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

cleanFlightsDF = flightsDF.where("ArrDelay is not null and DepDelay is not null and DepTime is not null and ArrTime is not null")\
                          .withColumn("ArrDelay", F.col("ArrDelay").cast(IntegerType()))\
                          .withColumn("DepDelay", F.col("DepDelay").cast(IntegerType()))\
                          .withColumn("ArrTime", F.col("ArrTime").cast(IntegerType()))\
                          .withColumn("DepTime", F.col("DepTime").cast(IntegerType())).cache()

# Dividimos en train y test de manera aleatoria usando la semilla 123 para los números aleatorios, para que sea reproducible. 
# Esto devuelve una lista de dos DataFrames. La división hará que el primer elemento de la lista sea un DF 
# con aprox. el 70 % de las filas. Lo usaremos para entrenar. El otro DF tendrá aprox el 30 % restante de las filas. 

splits = cleanFlightsDF.randomSplit([0.7, 0.3], seed = 123)
trainDF = splits[0].cache() # El primer DF tiene el 70 % de los datos
testDF = splits[1].cache()

# Definimos tres categorías: <15, entre 15 y 60, >60. Cada una se codifica con un número real: 0.0, 1.0, 2.0 
splitsDelays = [-float("inf"), 15, 60, float("inf")]
arrDelayBucketizer = Bucketizer(splits=splitsDelays, inputCol="ArrDelay", outputCol="ArrDelayBucketed")

# Definimos varias franjas: 00:00 - 06:00, 06:00 - 12:00, 12:00 - 18:00, 18:00 - 22:00, 22:00 - 00:00
splitsDepTime = [0, 600, 1200, 1800, 2200, 2400]
depTimeBucketizer = Bucketizer(splits=splitsDepTime, inputCol="DepTime", outputCol="DepTimeBucketed")

# Indexamos las columnas categóricas Origin, Dest y DayOfWeek, para traducirlas a reales con la parte decimal a 0.
# Recordemos que esto también introduce metadatos adicionales en la columna resultante para indicar que, aunque sea
# una columna de números reales, en realidad están representando categorías y debe ser tratada como tal por el algoritmo
indexers = StringIndexer(inputCols=["Origin", "Dest", "DayOfWeek"], 
                         outputCols=["OriginIndexed", "DestIndexed", "DayOfWeekIndexed"],
                         handleInvalid="keep")  # si tuviéramos que transformar una categoría no vista, se codifica con uno más de la última existente

# Ahora unimos todas las columnas que se usarán como variables en una sola columna de tipo vector llamada featuresVector
vectorAssembler = VectorAssembler(inputCols = ["DepDelay", "DepTimeBucketed", "OriginIndexed", "DestIndexed", "DayOfWeekIndexed"], 
                                  outputCol = "featuresVector")

randomForest = RandomForestClassifier(featuresCol = "featuresVector",
                                      labelCol = "ArrDelayBucketed",
                                      numTrees = 50, maxBins=360)  # maxBins debe ser al menos el número de categorías de la variable que tenga más categorías

pipeline = Pipeline(stages=[depTimeBucketizer, indexers, vectorAssembler, randomForest])

Aplicamos fit para ajustar el pipeline completo. Lo que esto hace es, para cada etapa:
- Si la etapa es un Transformer, invoca al método transform() pasándole el DF que recibe de la etapa anterior,
  y envía el resultado (DF transformado) a la etapa siguiente del pipeline.
- Si la etapa es un Estimator, invoca al método fit() pasándole como argumento el DF recibido de la etapa anterior,
  y después invoca transform() sobre el objeto devuelto por fit, pasando de nuevo dicho DF como argumento. 
  Finalmente el DF devuelvo por transform es enviado a la etapa siguiente del pipeline.

In [0]:
pipelineModel = pipeline.fit(trainDF)   # CUIDADO: esta línea tarda aproximadamente 15 minutos

#### Guardamos el pipeline entrenado en DBFS

El modelo entrenado no contiene datos y existe como un objeto de Python convencional. Al guardarlo, no se puede especificar directamente Azure DataLake Storage como destino, ya que la comunicación con ADLS se hace a través de Spark porque el cluster tiene habilitada la propiedad adecuada (mediante token) para permitirle la escritura, pero fuera de Spark no tenemos acceso excepto que hubiésemos montado un volumen en DBFS apuntando a una ubicación de ADLS.

In [0]:
pipelineModel.save("/user/flights_rf")

#### Mostramos la estructura de carpetas guardada en DBFS

In [0]:
!ls -l /dbfs/user/flights_rf
!ls -ls /dbfs/user/flights_rf/metadata
!cat /dbfs/user/flights_rf/metadata/part-00000

total 8
drwxrwxrwx 2 nobody nogroup 4096 Feb 19 22:21 metadata
drwxrwxrwx 2 nobody nogroup 4096 Feb 19 21:40 stages
total 1
0 -rwxrwxrwx 1 nobody nogroup   0 Feb 19 22:21 _SUCCESS
1 -rwxrwxrwx 1 nobody nogroup 328 Feb 19 22:21 part-00000
{"class":"org.apache.spark.ml.PipelineModel","timestamp":1708381289146,"sparkVersion":"3.5.0","uid":"PipelineModel_b3873eaeabaa","paramMap":{"stageUids":["Bucketizer_8d7ab4a234a8","Bucketizer_534ec1c39b7c","StringIndexer_d80f75967a21","VectorAssembler_30a026de2c3d","RandomForestClassifier_2d56a1d93e72"]},"defaultParamMap":{}}


In [0]:
!ls -ls /dbfs/user/flights_rf/stages
!cat /dbfs/user/flights_rf/stages/0_Bucketizer_8d7ab4a234a8/metadata/part-00000

!ls -l /dbfs/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/data

!ls -l /dbfs/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/treesMetadata/

total 20
4 drwxrwxrwx 2 nobody nogroup 4096 Feb 19 21:40 0_Bucketizer_8d7ab4a234a8
4 drwxrwxrwx 2 nobody nogroup 4096 Feb 19 21:40 1_Bucketizer_534ec1c39b7c
4 drwxrwxrwx 2 nobody nogroup 4096 Feb 19 21:40 2_StringIndexer_d80f75967a21
4 drwxrwxrwx 2 nobody nogroup 4096 Feb 19 21:40 3_VectorAssembler_30a026de2c3d
4 drwxrwxrwx 2 nobody nogroup 4096 Feb 19 22:21 4_RandomForestClassifier_2d56a1d93e72
{"class":"org.apache.spark.ml.feature.Bucketizer","timestamp":1708381289684,"sparkVersion":"3.5.0","uid":"Bucketizer_8d7ab4a234a8","paramMap":{"outputCol":"DepTimeBucketed","splits":[0.0,600.0,1200.0,1800.0,2200.0,2400.0],"inputCol":"DepTime"},"defaultParamMap":{"outputCol":"Bucketizer_8d7ab4a234a8__output","handleInvalid":"error"}}
total 66
-rwxrwxrwx 1 nobody nogroup     0 Feb 19 22:21 _SUCCESS
-rwxrwxrwx 1 nobody nogroup   124 Feb 19 22:21 _committed_1095758900130980631
-rwxrwxrwx 1 nobody nogroup     0 Feb 19 22:21 _started_1095758900130980631
-rwxrwxrwx 1 nobody nogroup 66260 Feb 19 22:21 

#### Contenido de la carpeta data: un fichero parquet

In [0]:
spark.read.parquet("dbfs:/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/data").display()

treeID,nodeData
0,"List(0, 0.0, 0.3066986532881446, List(1400679.0, 207708.0, 96122.0), 1704509, 0.12423724447177814, 1, 2, List(0, List(20.5), -1))"
0,"List(1, 0.0, 0.11509723414782012, List(1372656.0, 86364.0, 3066.0), 1462086, -1.0, -1, -1, List(-1, List(), -1))"
0,"List(2, 1.0, 0.5887438848373557, List(28023.0, 121344.0, 93056.0), 242423, 0.2702654737764169, 3, 10, List(0, List(67.5), -1))"
0,"List(3, 1.0, 0.4238071944788783, List(27996.0, 115964.0, 14329.0), 158289, 0.03731328048591487, 4, 9, List(0, List(29.5), -1))"
0,"List(4, 1.0, 0.4984614278790519, List(21349.0, 32354.0, 1052.0), 54755, 0.008596045977938338, 5, 6, List(2, List(2.0, 5.0, 23.0, 29.0, 30.0, 34.0, 36.0, 38.0, 40.0, 42.0, 54.0, 55.0, 58.0, 64.0, 66.0, 67.0, 68.0, 73.0, 76.0, 82.0, 84.0, 87.0, 94.0, 95.0, 97.0, 101.0, 102.0, 103.0, 107.0, 108.0, 109.0, 112.0, 115.0, 116.0, 119.0, 121.0, 122.0, 123.0, 124.0, 125.0, 127.0, 132.0, 133.0, 138.0, 143.0, 144.0, 145.0, 146.0, 149.0, 151.0, 152.0, 153.0, 154.0, 155.0, 156.0, 158.0, 161.0, 162.0, 166.0, 168.0, 169.0, 171.0, 172.0, 174.0, 175.0, 177.0, 178.0, 180.0, 181.0, 183.0, 185.0, 186.0, 188.0, 189.0, 191.0, 193.0, 196.0, 197.0, 198.0, 199.0, 201.0, 202.0, 203.0, 204.0, 205.0, 206.0, 207.0, 209.0, 211.0, 213.0, 214.0, 215.0, 216.0, 217.0, 218.0, 220.0, 222.0, 223.0, 224.0, 225.0, 226.0, 228.0, 229.0, 230.0, 231.0, 232.0, 233.0, 234.0, 236.0, 237.0, 238.0, 239.0, 240.0, 242.0, 244.0, 245.0, 246.0, 247.0, 248.0, 249.0, 250.0, 252.0, 253.0, 254.0, 255.0, 256.0, 257.0, 259.0, 260.0, 261.0, 262.0, 263.0, 264.0, 266.0, 269.0, 270.0, 271.0, 272.0, 273.0, 275.0, 276.0, 278.0, 279.0, 280.0, 283.0, 284.0, 285.0, 286.0, 287.0, 288.0, 289.0, 290.0, 291.0, 292.0, 293.0, 294.0, 296.0, 298.0, 299.0, 300.0, 301.0, 302.0, 303.0, 304.0, 305.0, 306.0, 307.0, 308.0, 309.0, 310.0, 312.0, 313.0, 314.0, 316.0, 317.0, 318.0, 320.0, 321.0, 322.0, 323.0, 324.0, 325.0, 327.0, 328.0, 331.0, 332.0, 333.0, 334.0, 335.0, 336.0, 338.0, 339.0, 340.0, 341.0, 343.0, 345.0, 350.0, 351.0, 352.0, 353.0, 354.0), 356))"
0,"List(5, 1.0, 0.4151839049162983, List(3420.0, 8952.0, 161.0), 12533, -1.0, -1, -1, List(-1, List(), -1))"
0,"List(6, 1.0, 0.5120335158372296, List(17929.0, 23402.0, 891.0), 42222, 0.010996378137548585, 7, 8, List(0, List(24.5), -1))"
0,"List(7, 0.0, 0.5159790605135011, List(10429.0, 9995.0, 345.0), 20769, -1.0, -1, -1, List(-1, List(), -1))"
0,"List(8, 1.0, 0.4865716179625706, List(7500.0, 13407.0, 546.0), 21453, -1.0, -1, -1, List(-1, List(), -1))"
0,"List(9, 1.0, 0.32727876512560805, List(6647.0, 83610.0, 13277.0), 103534, -1.0, -1, -1, List(-1, List(), -1))"


#### Contenido de la carpeta treesMetadata: otro fichero parquet

In [0]:
spark.read.parquet("dbfs:/user/flights_rf/stages/4_RandomForestClassifier_2d56a1d93e72/treesMetadata").display()

treeID,metadata,weights
0,"{""class"":""org.apache.spark.ml.classification.DecisionTreeClassificationModel"",""timestamp"":1708381294867,""sparkVersion"":""3.5.0"",""uid"":""dtc_5379679f9099"",""paramMap"":{""labelCol"":""ArrDelayBucketed"",""maxBins"":360,""featuresCol"":""featuresVector""},""defaultParamMap"":{""leafCol"":"""",""maxDepth"":5,""impurity"":""gini"",""minInfoGain"":0.0,""labelCol"":""label"",""maxMemoryInMB"":256,""probabilityCol"":""probability"",""maxBins"":32,""minWeightFractionPerNode"":0.0,""rawPredictionCol"":""rawPrediction"",""checkpointInterval"":10,""cacheNodeIds"":false,""predictionCol"":""prediction"",""featuresCol"":""features"",""minInstancesPerNode"":1,""seed"":-5387697053847413545}}",1.0
1,"{""class"":""org.apache.spark.ml.classification.DecisionTreeClassificationModel"",""timestamp"":1708381294868,""sparkVersion"":""3.5.0"",""uid"":""dtc_1c614cd62ae8"",""paramMap"":{""featuresCol"":""featuresVector"",""labelCol"":""ArrDelayBucketed"",""maxBins"":360},""defaultParamMap"":{""checkpointInterval"":10,""minInstancesPerNode"":1,""featuresCol"":""features"",""rawPredictionCol"":""rawPrediction"",""maxMemoryInMB"":256,""cacheNodeIds"":false,""leafCol"":"""",""seed"":-5387697053847413545,""maxDepth"":5,""probabilityCol"":""probability"",""impurity"":""gini"",""labelCol"":""label"",""predictionCol"":""prediction"",""minInfoGain"":0.0,""minWeightFractionPerNode"":0.0,""maxBins"":32}}",1.0
2,"{""class"":""org.apache.spark.ml.classification.DecisionTreeClassificationModel"",""timestamp"":1708381294868,""sparkVersion"":""3.5.0"",""uid"":""dtc_429536b150fa"",""paramMap"":{""maxBins"":360,""featuresCol"":""featuresVector"",""labelCol"":""ArrDelayBucketed""},""defaultParamMap"":{""maxMemoryInMB"":256,""maxBins"":32,""checkpointInterval"":10,""featuresCol"":""features"",""predictionCol"":""prediction"",""minWeightFractionPerNode"":0.0,""minInfoGain"":0.0,""impurity"":""gini"",""seed"":-5387697053847413545,""leafCol"":"""",""cacheNodeIds"":false,""maxDepth"":5,""labelCol"":""label"",""probabilityCol"":""probability"",""minInstancesPerNode"":1,""rawPredictionCol"":""rawPrediction""}}",1.0
3,"{""class"":""org.apache.spark.ml.classification.DecisionTreeClassificationModel"",""timestamp"":1708381294869,""sparkVersion"":""3.5.0"",""uid"":""dtc_14359d6ec5d0"",""paramMap"":{""featuresCol"":""featuresVector"",""maxBins"":360,""labelCol"":""ArrDelayBucketed""},""defaultParamMap"":{""featuresCol"":""features"",""minWeightFractionPerNode"":0.0,""probabilityCol"":""probability"",""leafCol"":"""",""minInstancesPerNode"":1,""checkpointInterval"":10,""minInfoGain"":0.0,""impurity"":""gini"",""rawPredictionCol"":""rawPrediction"",""maxDepth"":5,""maxBins"":32,""maxMemoryInMB"":256,""cacheNodeIds"":false,""labelCol"":""label"",""predictionCol"":""prediction"",""seed"":-5387697053847413545}}",1.0
4,"{""class"":""org.apache.spark.ml.classification.DecisionTreeClassificationModel"",""timestamp"":1708381294869,""sparkVersion"":""3.5.0"",""uid"":""dtc_f76f4c9e744b"",""paramMap"":{""maxBins"":360,""labelCol"":""ArrDelayBucketed"",""featuresCol"":""featuresVector""},""defaultParamMap"":{""seed"":-5387697053847413545,""maxBins"":32,""checkpointInterval"":10,""minInstancesPerNode"":1,""impurity"":""gini"",""maxMemoryInMB"":256,""minWeightFractionPerNode"":0.0,""maxDepth"":5,""minInfoGain"":0.0,""leafCol"":"""",""labelCol"":""label"",""predictionCol"":""prediction"",""cacheNodeIds"":false,""probabilityCol"":""probability"",""rawPredictionCol"":""rawPrediction"",""featuresCol"":""features""}}",1.0
5,"{""class"":""org.apache.spark.ml.classification.DecisionTreeClassificationModel"",""timestamp"":1708381294869,""sparkVersion"":""3.5.0"",""uid"":""dtc_c2bca1299672"",""paramMap"":{""labelCol"":""ArrDelayBucketed"",""featuresCol"":""featuresVector"",""maxBins"":360},""defaultParamMap"":{""maxDepth"":5,""impurity"":""gini"",""labelCol"":""label"",""minWeightFractionPerNode"":0.0,""rawPredictionCol"":""rawPrediction"",""leafCol"":"""",""cacheNodeI

### Predicciones con el pipeline entrenado

Vamos a hacer predicciones sobre los datos de test. Esto devolverá un nuevo DF al que se le han añadido varias columnas al final:
1. **`rawPrediction`**: magnitud que tiene una interpretación diferente según el algoritmo pero que intuitivamente nos da una
  medida de la confianza en cada posible clase (cuanto mayor, más confianza se tiene en que esa es la clase del ejemplo). En
  nuestro caso será un vector de 3 números reales puesto que nuestro problema tiene 3 clases
2. **`probability`**: vector de probabilidades de que el ejemplo pertenezca a cada una de las 3 clases de nuestro problema
3. **`prediction`**: clase para la cual la rawProbability es mayor.

## Explicar qué ocurre cuando no viene la columna ArrDelay en el DataFrame de datos para predicción

En un conjunto de datos reales a los cuales tengamos que hacerles predicciones, lo normal es que no tengamos ninguna columna target (ArrDelay) porque desconocemos ese dato y precisamente por eso querríamos predecirlo. Sin embargo, en nuestro pipeline hemos incluido la etapa `arrDelayBucketizer` que discretiza la variable ArrDelay. Esto significa que cualquier DataFrame que queramos predecir (`transform(...)`) con el pipeline entrenado tiene que tener una columna llamada ArrDelay para que la pieza `arrDelayBucketizer` no reviente y pueda funcionar con normalidad, **a pesar de que dicha columna ArrDelay no va a ser utilizada por nadie más en el pipeline a la hora de predecir**, ya que naturalmente, el RandomForestClassificationModel (modelo ya entrenado) no necesita ninguna variable target para calcular predicciones. Por tanto tenemos dos opciones:
* Incluir una variable ArrDelay ficticia, inventada, en el DataFrame de datos para predecir, con cualquier valor. La finalidad es simplemente **evitar** que la pieza `arrDelayBucketizer` (y por tanto, el pipeline completo) reviente al hacer predicciones.
* Quitar del pipeline la pieza `arrDelayBucketizer` y ajustarla **por fuera del pipeline**, antes incluso de hacer la división en train y test. De esta manera, como el pipeline no contendrá ninguna pieza que vaya a requerir una columna ArrDelay, no será problema que el DataFrame con el que predecimos no incluya esa columna.

In [0]:
predictionsDF = pipelineModel.transform(testDF)

predictionsDF.select("rawPrediction", "probability", "prediction").where("prediction = 1.0").show(truncate=False)

#pipelineModel.transform(realDataDF.withColumn("ArrDelay", F.lit(999)))

+----------------------------------------------------------+------------------------------------------------------------+----------+
|rawPrediction                                             |probability                                                 |prediction|
+----------------------------------------------------------+------------------------------------------------------------+----------+
|[20.756820607519334,25.319843072517155,3.9233363199635005]|[0.4151364121503867,0.5063968614503431,0.07846672639927002] |1.0       |
|[8.373379200488465,34.84936884325312,6.777251956258423]   |[0.16746758400976927,0.6969873768650623,0.13554503912516844]|1.0       |
|[9.347413915388794,34.6723367975612,5.980249287050016]    |[0.18694827830777586,0.693446735951224,0.11960498574100031] |1.0       |
|[8.568600912134917,34.65805421725962,6.773344870605469]   |[0.17137201824269832,0.6931610843451923,0.13546689741210938]|1.0       |
|[7.413165294960422,35.525851861027085,7.060982844012497]  |[0.148263

In [0]:
print("Hay {0} ejemplos de entrenamiento".format(trainDF.count()))

Hay 1705959 ejemplos de entrenamiento


In [0]:
predictionsDF.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- TaxiIn: string (nullable = true)
 |-- TaxiOut: string (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true)
 |-

### Evaluación del modelo (evaluamos las predicciones que hemos hecho sobre el DF de test)

In [0]:
# Select (prediction, true label) and compute test error
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Objeto evaluador, para evaluar las predicciones. Compara una columna de predicciones con la columna target del verdadero valor
# Hay varias métricas posibles, pero nosotros hemos elegido la más simple: porcentaje de acierto en clasificación.
evaluator = MulticlassClassificationEvaluator(
    labelCol="ArrDelayBucketed", predictionCol="prediction", metricName="accuracy")

accuracy = evaluator.evaluate(predictionsDF)

print("Test Error = %g " % (1.0 - accuracy))

Test Error = 0.0808506 


In [0]:
pipelineModel.stages[-1]  # La última etapa del pipeline entrenado es el algoritmo Random Forest entrenado

RandomForestClassificationModel: uid=RandomForestClassifier_2d56a1d93e72, numTrees=50, numClasses=3, numFeatures=5

In [0]:
pipelineModel.stages[-1].featureImportances

SparseVector(5, {0: 0.9851, 1: 0.0102, 2: 0.0021, 3: 0.0027})

### Ajuste de híper-parámetros utilizando Cross Validation sobre el subconjunto de train

In [0]:
randomForest.getOrDefault("numTrees")

50

### ATENCIÓN: la siguiente celda tarda unos 23 minutos en ejecutar

In [0]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Objeto ParamGrid al que le tenemos que indicar cada uno de los parámetros sobre los que queremos buscar,
# y la lista de valores posibles que queremos probar con cada uno. Deben ser parámetros tomados directamente
# del objeto estimador que se añadió al pipeline (no pueden ser de ningún otro modelo). En nuestro caso este objeto
# está almacenado en la variable randomForest que habíamos creado en celdas anteriores.
paramGrid = ParamGridBuilder()\
    .addGrid(randomForest.numTrees, [50, 150])\
    .addGrid(randomForest.maxDepth, [3, 5])\
    .build() # como tenemos 2 híper-parámetros con 2 valores posibles cada uno, hay 4 combinaciones posibles

# CrossValidator es un estimador. Cuando invocamos a fit(), probará todas las combinaciones de valores de los 
# parámetros, invocando con cada combinación al método fit() del objeto pipeline que le hemos pasado como argumento
crossval = CrossValidator(estimator=pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=3)

# Como hemos hecho 3 folds, habrá que entrenar 3 veces cada modelo candidato (cada combinación de parámetros)
# y obtener su media de accuracy. En total vamos a entrenar 12 modelos + 1 modelo extra al final 
# con todo el dataset de train (sin folds) usando la combinación de híper-parámetros que ya sabemos que funciona mejor

cvModel = crossval.fit(trainDF.sample(0.1, seed=123))  # intentamos evitar que tarde mucho en este ejemplo
cvModel.bestModel

PipelineModel_b18cd90ea13b

In [0]:
prediccionesMejorDF = cvModel.bestModel.transform(testDF)

#### El objeto RandomForestModel (modelo ajustado presente en la última etapa del pipeline ya ajustado) dispone de ciertos atajos para recuperar valores de algunos parámetros (por ejemplo getNumTrees) pero no de otros (por ejemplo, no existe getMaxDepth)

In [0]:
rf = cvModel.bestModel.stages[-1]
print("Número óptimo de árboles: {0}".format(rf.getNumTrees))
print("Max depth óptimo: {0}".format(rf.getOrDefault('maxDepth')))

Número óptimo de árboles: 50
Max depth óptimo: 3
